# Alaska Sheep Draw Area Trends

Run in this order for normal use:
1. Cell 2 (load data)
2. Cell 3 (define plotting function)
3. Cell 5 (main dropdown output)

Optional:
- Cell 6 to plot every area (many charts).


In [2]:
# CELL 2: Load and clean draw data
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Prefer local project file; fall back to absolute path if needed.
CSV_PATH = Path('sheep_draw_by_area_raw.csv')
if not CSV_PATH.exists():
    CSV_PATH = Path(r'C:\Users\mattr\OneDrive\Desktop\Visual Studio\Draw Results App\sheep_draw_by_area_raw.csv')

df = pd.read_csv(CSV_PATH)

numeric_cols = ['Year', 'Permits', 'Hunters', 'MaleHarvest', 'FemaleHarvest', 'TotalHarvest', 'SuccessPct']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['HuntCode'] = df['HuntCode'].astype(str).str.strip()
df = df.sort_values(['HuntCode', 'Year']).reset_index(drop=True)

min_year = int(df['Year'].min())
max_year = int(df['Year'].max())
all_years = list(range(min_year, max_year + 1))

print(f'Loaded {len(df)} rows across {df["HuntCode"].nunique()} draw areas.')
print(f'Year range: {min_year} to {max_year}')

Loaded 486 rows across 55 draw areas.
Year range: 2015 to 2025


In [3]:
# CELL 3: Plot function used by the dropdown
def plot_area_single_graph(area_code: str, data: pd.DataFrame, years: list[int]) -> None:
    area = data[data['HuntCode'] == area_code].copy()
    if area.empty:
        print(f'No data for area: {area_code}')
        return

    area = area.set_index('Year').reindex(years).reset_index()
    area['HuntCode'] = area_code

    for col in ['Permits', 'Hunters', 'MaleHarvest', 'FemaleHarvest', 'TotalHarvest']:
        area[col] = area[col].fillna(0)

    fig, ax1 = plt.subplots(figsize=(12, 6))
    ax1.plot(area['Year'], area['Permits'], marker='o', linewidth=2, label='Permits', color='#1f77b4')
    ax1.plot(area['Year'], area['Hunters'], marker='s', linewidth=2, label='Hunters', color='#2ca02c')
    ax1.plot(area['Year'], area['TotalHarvest'], marker='^', linewidth=2, label='Total Harvest', color='#d62728')
    ax1.set_xlabel('Year')
    ax1.set_ylabel('Counts (Permits/Hunters/Harvest)')
    ax1.set_xticks(years)
    ax1.tick_params(axis='x', rotation=45)
    ax1.grid(True, alpha=0.3)

    ax2 = ax1.twinx()
    ax2.plot(area['Year'], area['SuccessPct'], marker='D', linestyle='--', linewidth=2, label='Success %', color='#9467bd')
    ax2.set_ylabel('Success %')
    ax2.set_ylim(0, 100)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

    plt.title(f'Alaska Sheep Draw Area {area_code}: Opportunities, Harvest, and Success %')
    plt.tight_layout()
    plt.show()


## Interactive Unit Selector (Main Output)

Run Cell 5 after Cells 2 and 3.
This renders one dropdown and one output panel only.


In [ ]:
# CELL 5 (MAIN OUTPUT): Interactive dropdown + draw links
import re
import requests
import ipywidgets as widgets
from bs4 import BeautifulSoup
from IPython.display import display, HTML, clear_output

# Remove previous widget instances when rerunning this cell.
if '_hunt_selector_box' in globals() and _hunt_selector_box is not None:
    try:
        _hunt_selector_box.close()
    except Exception:
        pass
if '_hunt_selector' in globals() and _hunt_selector is not None:
    try:
        _hunt_selector.close()
    except Exception:
        pass
if '_hunt_output' in globals() and _hunt_output is not None:
    try:
        _hunt_output.close()
    except Exception:
        pass

# Clear the existing cell display so reruns do not stack output blocks.
clear_output(wait=True)

OPPORTUNITIES_URL = 'https://www.adfg.alaska.gov/index.cfm?adfg=sheephunting.opportunities&hunttype=D&performsearch=true'
GMU_INFO_BASE = 'https://www.adfg.alaska.gov/index.cfm?adfg=huntingmaps.gmuinfo&gmu={gmu}'

HUNTFILE_ID_BY_CODE = {
    'DS003': '28062', 'DS004': '27405', 'DS026': '28060', 'DS033': '27406', 'DS065': '28011',
    'DS102': '28010', 'DS123': '28039', 'DS124': '28034', 'DS125': '28037', 'DS126': '28030',
    'DS130': '28019', 'DS131': '28018', 'DS132': '28023', 'DS134': '28021', 'DS135': '28038',
    'DS136': '28009', 'DS137': '28017', 'DS138': '28006', 'DS140': '28007', 'DS141': '28013',
    'DS156': '27399', 'DS160': '28031', 'DS165': '28036', 'DS170': '28024', 'DS175': '28026',
    'DS180': '28043', 'DS185': '28012', 'DS190': '28032', 'DS195': '28022', 'DS196': '28020',
    'DS197': '28028', 'DS198': '28016', 'DS203': '28029', 'DS204': '28040', 'DS205': '28025',
    'DS206': '27068', 'DS207': '28015', 'DS224': '28063', 'DS225': '28004', 'DS230': '28014',
    'DS231': '27101', 'DS232': '28005', 'DS233': '28064', 'DS236': '28042', 'DS237': '27093',
    'DS240': '28035', 'DS241': '28044', 'DS260': '28027', 'DS265': '28041', 'DS270': '28033',
    'DS285': '28008'
}

def draw_details_url(hunt_code: str, huntfile_id: str = '') -> str:
    hid = huntfile_id or HUNTFILE_ID_BY_CODE.get(hunt_code, '')
    if not hid:
        return ''
    return f'https://www.adfg.alaska.gov/index.cfm?adfg=sheephunting.hunt_details&huntfile_id={hid}'

def fetch_hunt_unit_lookup() -> dict:
    codes = sorted(df['HuntCode'].dropna().unique().tolist())
    lookup = {
        code: {
            'unit_text': 'ADF&G page unavailable; using local data.',
            'gmus': [],
            'huntfile_id': HUNTFILE_ID_BY_CODE.get(code, ''),
            'details_url': draw_details_url(code),
        }
        for code in codes
    }

    try:
        resp = requests.get(OPPORTUNITIES_URL, timeout=25, headers={'User-Agent': 'Mozilla/5.0'})
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text or '', 'html.parser')

        for tr in soup.find_all('tr'):
            cells = [c.get_text(' ', strip=True) for c in tr.find_all(['td', 'th'])]
            if len(cells) < 2:
                continue

            hunt_code = cells[0].strip().upper()
            if not re.fullmatch(r'DS\d{3}', hunt_code):
                continue

            unit_text = cells[1].strip()
            gmus = re.findall(r'\b\d{1,2}\b', unit_text)
            huntfile_id = HUNTFILE_ID_BY_CODE.get(hunt_code, '')

            for a in tr.find_all('a', href=True):
                href = a['href'].strip().lower()
                if 'hunt_details' in href and 'huntfile_id' in href:
                    m = re.search(r'huntfile_id=(\d+)', href)
                    if m:
                        huntfile_id = m.group(1)
                    break

            lookup[hunt_code] = {
                'unit_text': unit_text,
                'gmus': gmus,
                'huntfile_id': huntfile_id,
                'details_url': draw_details_url(hunt_code, huntfile_id),
            }
    except Exception:
        pass

    return lookup

def _render_selected_hunt_content(hunt_code: str, hunt_lookup: dict, output_widget: widgets.Output) -> None:
    with output_widget:
        output_widget.clear_output(wait=True)
        plot_area_single_graph(hunt_code, df, all_years)

        info = hunt_lookup.get(hunt_code, {
            'unit_text': 'Not found on opportunities page.',
            'gmus': [],
            'huntfile_id': '',
            'details_url': '',
        })

        unit_text = info.get('unit_text', 'Not available')
        gmus = info.get('gmus', [])
        huntfile_id = info.get('huntfile_id', '')
        details_url = info.get('details_url', '') or draw_details_url(hunt_code, huntfile_id)

        display(HTML(f'<h3>ADF&G Unit Info for {hunt_code}</h3>'))
        display(HTML(f'<p><b>Unit/Area:</b> {unit_text}</p>'))

        links = []
        if details_url:
            links.append((f'ADF&G draw details for {hunt_code}', details_url))
        else:
            links.append((f'ADF&G draw opportunities for {hunt_code}', OPPORTUNITIES_URL))

        if huntfile_id:
            links.append((
                f'ADF&G maps/hunt details for {hunt_code}',
                f'https://www.adfg.alaska.gov/index.cfm?adfg=huntingmaps.hunt_details&huntfile_id={huntfile_id}',
            ))

        for gmu in gmus:
            links.append((f'Open official GMU {gmu} map/info', GMU_INFO_BASE.format(gmu=gmu)))

        links_html = ''.join([
            f"<li style='margin: 6px 0;'><a href='{href}' target='_blank' rel='noopener noreferrer' style='background:#e8f0fe; color:#0b57d0; font-weight:bold; padding:6px 10px; border-radius:5px; display:inline-block; text-decoration:none; border:1px solid #b7cdfa;'>{label}</a></li>"
            for label, href in links
        ])

        display(HTML(f"<p><b>Resolved Draw URL:</b> <a href='{details_url or OPPORTUNITIES_URL}' target='_blank' rel='noopener noreferrer'>{details_url or OPPORTUNITIES_URL}</a></p>"))
        display(HTML(f"<p style='margin-top:12px;'><b>Open selected unit:</b></p><ul style='padding-left:18px; margin-top:4px;'>{links_html}</ul>"))

        if gmus:
            display(HTML(f"<p><b>GMU(s):</b> {', '.join(gmus)}</p>"))

hunt_lookup = fetch_hunt_unit_lookup()
available_codes = sorted(df['HuntCode'].dropna().unique().tolist())

_hunt_selector = widgets.Dropdown(
    options=available_codes,
    value=available_codes[0] if available_codes else None,
    description='Hunt Code:',
    layout=widgets.Layout(width='320px'),
)
_hunt_output = widgets.Output()
_hunt_selector_box = widgets.VBox([_hunt_selector, _hunt_output])

def _on_hunt_change(change):
    if change.get('name') == 'value' and change.get('new'):
        _render_selected_hunt_content(change['new'], hunt_lookup, _hunt_output)

_hunt_selector.observe(_on_hunt_change, names='value')
display(_hunt_selector_box)
if _hunt_selector.value:
    _render_selected_hunt_content(_hunt_selector.value, hunt_lookup, _hunt_output)


In [ ]:
# CELL 6 (OPTIONAL): Plot every draw area (many charts)
for code in sorted(df['HuntCode'].dropna().unique()):
    plot_area_single_graph(code, df, all_years)
